# Projeto Integrador — Selene (Data Science: Entrega 2)
**Aluno:** Messias Rafael Bernardes  
**Disciplina:** Data Science & Desenvolvimento Móvel  

---
## 1. Governança e Ética de Dados
- **Fonte de Referência:** Painel Público da Ouvidoria Nacional de Direitos Humanos (Ligue 180 — MDHC) e Atlas da Violência (IPEA).
- **Licença de Uso:** Dados Abertos Governamentais (Decreto nº 8.777/2016).
- **Privacidade e LGPD:** Todos os registros analisados são anonimizados e isentos de dados pessoais sensíveis que permitam identificar vítimas reais.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dados/denuncias_ligue180.csv')
print(f'Dimensões iniciais da base bruta: {df.shape[0]} linhas x {df.shape[1]} colunas\n')
print(df.head().to_string())


Dimensões iniciais da base bruta: 306 linhas x 10 colunas

   id_registro  idade_vitima  episodios_anteriores  tempo_convivencia_anos  dias_ate_denuncia tipo_violencia relacao_agressor local_ocorrencia turno_ocorrencia  risco_grave
0         1001          38.0                   2.0                     1.0               12.0    Patrimonial         Familiar      Via Pública            Noite            0
1         1002          32.0                   2.0                     6.8               12.0         Física      Companheiro       Residência            Noite            1
2         1003          39.0                   5.0                     2.5               16.0         Sexual   Ex-companheiro       Residência            Noite            1
3         1004          46.0                   6.0                     0.8                2.0    Patrimonial   Ex-companheiro       Residência            Noite            1
4         1005          32.0                  16.0                     5.6  

## 2. Qualidade dos Dados (Limpeza, Valores Ausentes e Normalização)
Verificação inicial de registros duplicados, categorias inconsistentes e valores nulos (`NaN`).


In [2]:
duplicados_antes = int(df.duplicated().sum())
print(f'Total de registros duplicados encontrados: {duplicados_antes}\n')
print('Valores ausentes (nulos) por coluna antes do tratamento:')
print(df.isnull().sum().to_string())


Total de registros duplicados encontrados: 6

Valores ausentes (nulos) por coluna antes do tratamento:
id_registro               0
idade_vitima              8
episodios_anteriores      5
tempo_convivencia_anos    0
dias_ate_denuncia         0
tipo_violencia            4
relacao_agressor          0
local_ocorrencia          0
turno_ocorrencia          0
risco_grave               0


In [3]:
df_clean = df.drop_duplicates().copy()

df_clean['tipo_violencia'] = (
    df_clean['tipo_violencia']
    .astype(str)
    .str.strip()
    .replace({'nan': np.nan, 'None': np.nan, 'psicologica': 'Psicológica', 'FISICA': 'Física'})
)

df_clean['idade_vitima'] = df_clean['idade_vitima'].fillna(df_clean['idade_vitima'].median())
df_clean['episodios_anteriores'] = df_clean['episodios_anteriores'].fillna(df_clean['episodios_anteriores'].median())
df_clean['tipo_violencia'] = df_clean['tipo_violencia'].fillna(df_clean['tipo_violencia'].mode()[0])

print(f'Dimensões após remoção de duplicatas e limpeza: {df_clean.shape[0]} linhas x {df_clean.shape[1]} colunas')
print(f'Duplicados restantes: {int(df_clean.duplicated().sum())}\n')
print('Valores ausentes após o tratamento:')
print(df_clean.isnull().sum().to_string())


Dimensões após remoção de duplicatas e limpeza: 300 linhas x 10 colunas
Duplicados restantes: 0

Valores ausentes após o tratamento:
id_registro               0
idade_vitima              0
episodios_anteriores      0
tempo_convivencia_anos    0
dias_ate_denuncia         0
tipo_violencia            0
relacao_agressor          0
local_ocorrencia          0
turno_ocorrencia          0
risco_grave               0


In [4]:
num_cols = ['idade_vitima', 'episodios_anteriores', 'tempo_convivencia_anos', 'dias_ate_denuncia']
df_norm = df_clean.copy()

for col in num_cols:
    c_min = df_clean[col].min()
    c_max = df_clean[col].max()
    df_norm[col + '_norm'] = ((df_clean[col] - c_min) / (c_max - c_min)).round(4)

print('Amostra das variáveis numéricas normalizadas (Min-Max [0, 1]):\n')
print(df_norm[[c + '_norm' for c in num_cols]].head().to_string())


Amostra das variáveis numéricas normalizadas (Min-Max [0, 1]):

   idade_vitima_norm  episodios_anteriores_norm  tempo_convivencia_anos_norm  dias_ate_denuncia_norm
0             0.4167                     0.0526                       0.0255                  0.1348
1             0.2917                     0.0526                       0.3214                  0.1348
2             0.4375                     0.2105                       0.1020                  0.1798
3             0.5833                     0.2632                       0.0153                  0.0225
4             0.2917                     0.7895                       0.2602                  0.1685


## 3. Análise Exploratória de Dados (EDA)
### 3.1 Medidas de Tendência Central (Média, Mediana e Moda)


In [5]:
tendencia_central = pd.DataFrame({
    'Média': df_clean[num_cols].mean().round(2),
    'Mediana': df_clean[num_cols].median().round(2),
    'Moda': [df_clean[c].mode()[0] for c in num_cols]
})
print('Medidas de Tendência Central (Média, Mediana e Moda):\n')
print(tendencia_central.to_string())


Medidas de Tendência Central (Média, Mediana e Moda):

                        Média  Mediana  Moda
idade_vitima            33.65    34.00  34.0
episodios_anteriores     4.30     3.00   2.0
tempo_convivencia_anos   5.61     4.95   3.0
dias_ate_denuncia       11.90     9.00   1.0


### 3.2 Medidas de Dispersão (Desvio Padrão, Variância, Amplitude e IQR)


In [6]:
q1 = df_clean[num_cols].quantile(0.25)
q3 = df_clean[num_cols].quantile(0.75)

dispersao = pd.DataFrame({
    'Desvio Padrão': df_clean[num_cols].std().round(2),
    'Variância': df_clean[num_cols].var().round(2),
    'Mínimo': df_clean[num_cols].min().round(2),
    'Máximo': df_clean[num_cols].max().round(2),
    'Amplitude': (df_clean[num_cols].max() - df_clean[num_cols].min()).round(2),
    'IQR (Q3 - Q1)': (q3 - q1).round(2)
})
print('Medidas de Dispersão (Desvio Padrão, Variância, Amplitude e IQR):\n')
print(dispersao.to_string())


Medidas de Dispersão (Desvio Padrão, Variância, Amplitude e IQR):

                        Desvio Padrão  Variância  Mínimo  Máximo  Amplitude  IQR (Q3 - Q1)
idade_vitima                     8.10      65.63    18.0    66.0       48.0          11.00
episodios_anteriores             3.51      12.31     1.0    20.0       19.0           4.00
tempo_convivencia_anos           3.60      12.93     0.5    20.1       19.6           4.22
dias_ate_denuncia               12.17     148.06     0.0    89.0       89.0          12.00


### 3.3 Distribuição de Frequências
Análise de frequência absoluta e relativa (%) por tipo de violência, vínculo com o agressor e local da ocorrência.


In [7]:
freq_violencia = pd.DataFrame({
    'Frequência Absoluta': df_clean['tipo_violencia'].value_counts(),
    'Frequência Relativa (%)': (df_clean['tipo_violencia'].value_counts(normalize=True) * 100).round(2)
})

freq_agressor = pd.DataFrame({
    'Frequência Absoluta': df_clean['relacao_agressor'].value_counts(),
    'Frequência Relativa (%)': (df_clean['relacao_agressor'].value_counts(normalize=True) * 100).round(2)
})

freq_local = pd.DataFrame({
    'Frequência Absoluta': df_clean['local_ocorrencia'].value_counts(),
    'Frequência Relativa (%)': (df_clean['local_ocorrencia'].value_counts(normalize=True) * 100).round(2)
})

print('=== Distribuição de Frequências: Tipo de Violência ===')
print(freq_violencia.to_string())
print('\n=== Distribuição de Frequências: Relação com o Agressor ===')
print(freq_agressor.to_string())
print('\n=== Distribuição de Frequências: Local da Ocorrência ===')
print(freq_local.to_string())


=== Distribuição de Frequências: Tipo de Violência ===
                Frequência Absoluta  Frequência Relativa (%)
tipo_violencia                                              
Psicológica                     122                    40.67
Física                           86                    28.67
Moral                            48                    16.00
Patrimonial                      24                     8.00
Sexual                           20                     6.67

=== Distribuição de Frequências: Relação com o Agressor ===
                  Frequência Absoluta  Frequência Relativa (%)
relacao_agressor                                              
Companheiro                       151                    50.33
Ex-companheiro                     87                    29.00
Familiar                           47                    15.67
Outro                              15                     5.00

=== Distribuição de Frequências: Local da Ocorrência ===
                   Fr

## 4. Conclusões da Análise Exploratória para o Aplicativo Selene
1. **Predominância de Violência Psicológica e Física (`70%+`):** Mostra a importância de canais rápidos de ajuda antes do agravamento dos episódios.
2. **Ocorrências no Ambiente Doméstico (`~74%` na Residência) por Parceiros/Ex-parceiros (`~82%`):** Confirma a necessidade crítica do **mecanismo de disfarce (Calendário Menstrual)** do Selene, já que a vítima convive no mesmo ambiente que o agressor.
3. **Reincidência antes da Denúncia (Média de ~4 episódios anteriores):** O histórico de múltiplos episódios antes do primeiro acionamento servirá como base para o indicador de risco que será modelado na Entrega 3.
